In [1]:
import numpy as np
import math
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import time
from numpy.linalg import inv

import cvxpy as cvx
import itertools

In [2]:
from math import factorial as fac
def comb(x, y):
    try:
        return fac(x) // fac(y) // fac(x - y)
    except ValueError:
        return 0

#check if is an allowed state
def is_allowed_state(n_node, state):
    allowed = True

    for i in range(n_node):
        if state[i][i]>0:
            allowed = False
            break
            
        c1 = 0
        for j in range(i,n_node):
            if state[i][j]>0:
                c1+=1
        if c1>1:
            allowed = False
            break
            
    if allowed:
        for j in range(n_node):
            c2 = 0
            for i in range(j):
                if state[i][j]>0:
                    c2+=1
            if c2>1:
                allowed = False
                break
                
    if allowed:
        for i in range(n_node):
            for j in range(i+2, n_node):
                if state[i][j] > 0 and allowed:
                    if state[i][i+1]>0 or state[i][j-1]:
                        allowed = False
                    
                    for k in range(i+1,j):
                        if allowed==False:
                            break
                        for l in range(j+1,n_node):
                            if state[k][l]>0:
                                allowed=False
                                break
                        
                if not allowed:
                    break
            if not allowed:
                break
                
    return(allowed)   

# check if allowed action
def is_allowed_action(n_node, action):
    allowed = True
    
    if action[0][0] == 1 or action[n_node-1][n_node-1] == 1:
        return False
    
    for i in range(1,n_node-1):
            if action[i][i]>0:
                if action[i-1][i]==1 or action[i][i+1]==1:
                    allowed = False
                    break
                    
    if allowed:
        for i in range(n_node):
            for j in range(i+2,n_node):
                if action[i][j]>0:
                    allowed = False
                    break
            if not allowed:
                break
    
    return allowed  


def get_state_index(n_node, m_star, state):
    index = 0
    n_max = comb(n_node,2) #total number of links (physical + virtual)
    n = 0
    for i in range(n_node):
        for j in range(i+1,n_node):
            n = n+1
            index = index + state[i][j]*(m_star+1)**(n_max - n)
    return int(index)


def get_state_from_index(n_node, m_star, index):
    base = m_star+1
    base_num = ""
    while index>0:
        dig = int(index%base)
        base_num += str(dig)
        index //= base
    base_num = base_num[::-1]
    a = np.zeros([comb(n_node,2)])
    pad = comb(n_node,2)-len(base_num)
    for k in range(comb(n_node,2)):
        if k < pad:
            a[k] = 0
        else:
            a[k] = int(base_num[k-pad])
            
    state = np.zeros([n_node,n_node])
    k = 0
    for i in range(n_node):
        for j in range(i+1,n_node):
            k = k + 1
            state[i][j] = a[k-1]
            state[j][i] = state[i][j]
            
    return state


def get_action_index(n_node, state):
    index = 0
    n_max = 2*n_node - 1 #total number of physical links + number of nodes
    n = 0
    for i in range(n_node):
        for j in range(i,i+2):
            if j<n_node:
                n = n+1
                index = index + state[i][j]*(2)**(n_max - n)
    return int(index)


def get_action_from_index(n_node, index):
    base = 2
    base_num = ""
    while index>0:
        dig = int(index%base)
        base_num += str(dig)
        index //= base
    base_num = base_num[::-1]
    
    a = np.zeros([2*n_node - 1])
    pad = 2*n_node - 1 - len(base_num)
    for k in range(2*n_node - 1):
        if k < pad:
            a[k] = 0
        else:
            a[k] = int(base_num[k-pad])
            
    action = np.zeros([n_node,n_node])
    k = 0
    for i in range(n_node):
        for j in range(i,i+2):
            if j<n_node:
                k = k + 1
                action[i][j] = a[k-1]
                action[j][i] = action[i][j]
            
    return action


#define a function that determines if the specified location is a terminal state
def is_terminal_state(n_node,current_state):
  #if the state has a link between 1st and last node then it is terminal
    if current_state[0][n_node-1] > 0 and is_allowed_state(n_node,current_state):  
        return True
    else:
        return False


def get_allowed_states_and_actions(n_node,m_star,index=True,terminal=True):
    
    # Last modified: 8 January 2023
    
    '''
    Define the shape of the environment (i.e., its states). States for each node are 0,1,2,...,m_star where
    0 = inactive memory, 1 to m_star = active memory
    
    Number of states per elementary link = m_star + 1
    
    n_node is the number of nodes.
    
    if terminal=False, then get only the non-terminal/transient states.
    
    '''
    
    state = np.zeros([n_node, n_node])
    n_state = (m_star+1)**(comb(n_node,2))  # Total number of states in the system
    n_action = (2)**(2*n_node - 1)
    # 0 = wait for photon/BM, 1 = request for photon/BM (on nodes we do BM, on physical links entangled pair request)
    action = np.zeros([n_node, n_node])
    
    allowed_states = []
    allowed_actions = []
    for i in range(n_state):
        state=get_state_from_index(n_node, m_star, i)
        if not terminal:
            if is_allowed_state(n_node,state) and not is_terminal_state(n_node,state):
                if index:
                    allowed_states.append(i)
                else:
                    allowed_states.append(state)
            else:
                continue
        else:
            if is_allowed_state(n_node, state):
                if index:
                    allowed_states.append(i)       
                else:
                    allowed_states.append(state)

    for i in range(n_action):
        action=get_action_from_index(n_node, i)
        if is_allowed_action(n_node, action):
            if index:
                allowed_actions.append(i)   
            else:
                allowed_actions.append(action)

    allowed_states = np.array(allowed_states)
    allowed_actions = np.array(allowed_actions)
    
    return allowed_states, allowed_actions


# def get_initial_state_vector(n_node,m_star,p_l,terminal=False):
    
#     # Last modified: 8 January 2023
    
#     '''
#     By default, this excludes the terminal states. So the output is a vector
#     consising only of the non-terminal/transient states.
#     '''
    
#     n_links=n_node-1
#     allowed_state_indices=get_allowed_states_and_actions(n_node,m_star,index=True,terminal=terminal)[0]
#     n_allowed_states=len(allowed_state_indices)
    
#     L=[(i,i+1) for i in range(1,n_node)]
    
#     v=np.zeros(n_allowed_states)
#     #v={}
#     #for index in allowed_state_indices:
#     #    v[index]=0
    
#     S={}
#     S[(0,0)]=np.zeros((n_node,n_node))
    
#     # For zero active elementary links
#     v[get_state_index(n_node,m_star,S[(0,0)])]=(1-p_l)**(n_links)
    
#     # For one active elementary link
#     for l in L:
#         M=np.zeros((n_node,n_node))
#         M[l[0]-1,l[1]-1]=1
#         M[l[1]-1,l[0]-1]=1
#         S[l]=M
#         v[np.where(allowed_state_indices==get_state_index(n_node,m_star,S[l]))[0][0]]=p_l*(1-p_l)**(n_links-1)
    
#     # For 2 active elementary links up to n_links active elementary links
#     for n_active in range(2,n_links+1):
#         C=list(itertools.combinations(L,n_active))
#         for comb in C:
#             S[comb]=np.sum([S[comb[i]] for i in range(len(comb))],0)
#             v[np.where(allowed_state_indices==get_state_index(n_node,m_star,S[comb]))[0][0]]=p_l**(n_active)*(1-p_l)**(n_links-n_active)
    
#     return v#,list(S.values())

def get_initial_state_vector(n_node,m_star,p_l,terminal=False):
    
    # Last modified: 8 January 2023
    
    '''
    By default, this excludes the terminal states. So the output is a vector
    consising only of the non-terminal/transient states.
    '''
    
    n_links=n_node-1
    allowed_state_indices=get_allowed_states_and_actions(n_node,m_star,index=True,terminal=terminal)[0]
    n_allowed_states=len(allowed_state_indices)
    
    L=[(i,i+1) for i in range(1,n_node)]
    
    v=np.zeros(n_allowed_states)
    #v={}
    #for index in allowed_state_indices:
    #    v[index]=0
    
    S={}
    S[(0,0)]=np.zeros((n_node,n_node))
    
    # For zero active elementary links
    #v[get_state_index(n_node,m_star,S[(0,0)])]=(1-p_l)**(n_links)
    v[get_state_index(n_node,m_star,S[(0,0)])]=1

    # For one active elementary link
    for l in L:
        M=np.zeros((n_node,n_node))
        M[l[0]-1,l[1]-1]=1
        M[l[1]-1,l[0]-1]=1
        S[l]=M
        #v[np.where(allowed_state_indices==get_state_index(n_node,m_star,S[l]))[0][0]]=p_l*(1-p_l)**(n_links-1)
        v[np.where(allowed_state_indices==get_state_index(n_node,m_star,S[l]))[0][0]]=0

    # For 2 active elementary links up to n_links active elementary links
    for n_active in range(2,n_links+1):
        C=list(itertools.combinations(L,n_active))
        for comb in C:
            S[comb]=np.sum([S[comb[i]] for i in range(len(comb))],0)
            #v[np.where(allowed_state_indices==get_state_index(n_node,m_star,S[comb]))[0][0]]=p_l**(n_active)*(1-p_l)**(n_links-n_active)
            v[np.where(allowed_state_indices==get_state_index(n_node,m_star,S[comb]))[0][0]]=0
    
    return v



In [3]:
allowed_states,allowed_actions=get_allowed_states_and_actions(5,3,index=False,terminal=True)
allowed_states_indices,allowed_actions_indices=get_allowed_states_and_actions(5,3,index=True,terminal=False)

In [14]:
len(allowed_actions)

34

In [15]:
len(allowed_states)

562

In [8]:
get_state_index(3,3,np.array([[0,0,0],[0,0,0],[0,0,0]]))

0

In [9]:
get_initial_state_vector(3,3,0.3)

array([0.49, 0.21, 0.  , 0.  , 0.21, 0.09, 0.  , 0.  , 0.  , 0.  , 0.  ,
       0.  , 0.  , 0.  , 0.  , 0.  ])

In [4]:
# Train the Model
# Our next task is for our AI agent to learn about its environment by implementing a Q-learning model. 
# The learning process will follow these steps:
# Choose a random, non-terminal state for the agent to begin this new episode.
# Choose an action for the current state. Actions will be chosen using an epsilon greedy algorithm. 
# This algorithm will usually choose the most promising action for the AI agent, 
# but it will occasionally choose a less promising option in order to encourage the agent to explore the environment.
# Perform the chosen action, and transition to the next state (i.e., move to the next location).
# Receive the reward for moving to the new state, and calculate the temporal difference.
# Update the Q-value for the previous state and action pair.
# If the new (current) state is a terminal state, go to #1. Else, go to #2.
# This entire process will be repeated across 1000 episodes. 
# This will provide the AI agent sufficient opportunity to learn the shortest paths 
# Define Helper Functions




#ct=0
#cta=0
#allowed_states = []
#allowed_actions = []
#for i in range(n_state):
#    if is_allowed_state(n_node,get_state_from_index(n_node,m_star,i)):
#        ct+=1
#        allowed_states.append(i)       

#for i in range(n_action):
#    if is_allowed_action(n_node,get_action_from_index(n_node,i)):
#        cta+=1
#        allowed_actions.append(i)       
#allowed_states = np.array(allowed_states)
#allowed_actions = np.array(allowed_actions)

#q_values = np.zeros([ct,cta])
#print([ct,cta])

def get_random_state():
    i = np.random.randint(0,ct)
    return get_state_from_index(allowed_states[i])

def get_random_action():
    i = np.random.randint(0,cta)
    return get_action_from_index(allowed_actions[i])

#define a function that will choose a random, non-terminal starting state
def get_starting_state():
    
#   get a random state
    current_state = get_random_state()

#   continue choosing random state until a non-terminal state is identified
    while is_terminal_state(current_state):
        current_state = get_random_state()

    return current_state


#define an epsilon greedy algorithm that will choose which action to take next
def get_next_action(current_state, epsilon):
  #if a randomly chosen value between 0 and 1 is less than epsilon, 
  #then choose the most promising value from the Q-table for this state.
    state_index = get_state_index(current_state)
    
    if np.random.random() < epsilon:
        action_index = np.argmax(q_values[state_index])
        opt_action = get_action_from_index(action_index)
        #return opt_action
        if is_allowed_action(opt_action): 
            return opt_action
        else:
            return get_random_action()
    else: #choose a random action
        return get_random_action()

#define a function that will get the next location based on the chosen action
def get_next_state(n_node, m_star, current_state, action, p_l, p_bm):
    
    new_state = np.zeros([n_node,n_node])
    
    for i in range(n_node):
        for j in range(n_node):
            new_state[i][j] = current_state[i][j]
    
    flagg=0
    for i in range(n_node-1):
        if action[i][i+1]>0:
            flagg=1
            break
            
    if flagg==1:
    # wait               
        for i in range(n_node-1):
            for j in range(i+1,n_node):
                if action[i][j] == 0: #wait
                    if new_state[i][j] > 0:
                        new_state[i][j] = (new_state[i][j] + 1)%(m_star+1)
                        new_state[j][i] = new_state[i][j]
                       
    for i in range(n_node-1):
        j=i+1
        if action[i][j] == 1: #request entanglement
            for k in range(j-1):
                if current_state[k][j]>0:
                    new_state[k][j] = 0
                    new_state[j][k] = 0
        
            for k in range(j+2,n_node):
                if current_state[i][k]>0:
                    new_state[k][j] = 0
                    new_state[j][k] = 0
            
            if np.random.random()<=p_l:
                new_state[i][j] = 1
            else:
                new_state[i][j] = 0    
                
        

        new_state[j][i] = new_state[i][j]
        
        
    
    
    #Bell measurements

    for i in range(1,n_node-1):

        if action[i][i] == 1:  # request BM
            flag1 = 0; flag2 = 0;
            for k in range(i+1,n_node):
                if new_state[i][k] > 0:
                    node1 = k
                    node1_val = new_state[i][k]
                    flag1 = 1
                    break
            for l in range(i):
                if new_state[l][i] > 0:
                    node2 = l
                    node2_val = new_state[l][i]
                    flag2 = 1
                    break

            if flag1==1 and flag2==1:
                if np.random.random()<p_bm: #BM success
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = max(node1_val,node2_val)
                else:                       #BM failure
                    new_state[i][node1] = 0
                    new_state[node2][i] = 0
                    new_state[node2][node1] = 0
            else:
                if flag1==1 and flag2==0:
                    new_state[i][node1] = 0
                if flag2==1 and flag1==0:
                    new_state[node2][i] = 0
                    
        for i in range(n_node):
            for j in range(i+1,n_node):
                new_state[j][i] = new_state[i][j]
                
            
    return new_state

In [6]:
#### Transition matrices
from joblib import Parallel, delayed

def get_transition_probs(n_node,m_star,state,action,p_l,p_bm,terminal=True,trials=1000,states=0):
    
    # Last modified: 8 January 2023
    
    n_state = (m_star+1)**(comb(n_node,2))
    
    if type(states)==int:
        allowed_states=get_allowed_states_and_actions(n_node,m_star,index=True,terminal=True)[0]
    else:
        allowed_states=states
    #n_state=len(allowed_states)
    
    states = []

    for i in range(trials):
        state_next = get_next_state(n_node,m_star,state,action,p_l,p_bm)
        #print(a)
        states.append(get_state_index(n_node,m_star,state_next))
       
    state_freq = np.zeros(n_state)
    
    for state_index in allowed_states:
        for k in states:
            if k==state_index:
                state_freq[k]+=1
    #print(state_freq[np.where(state_freq>0)])
    return(state_freq/trials)


def get_transition_matrices(n_node,m_star,p_l,p_bm,terminal=False,trials=1000,parallel=True):
    n_jobs = 18
    # Last modified: 8 January 2023
    
    '''
    By putting terminal=False, this function will, by default, only give the sub-block of the transition
    matrices that correspond to the non-terminal/transient states. This is all that we need for the
    waiting time linear program below.
    '''
    
    allowed_states,allowed_actions=get_allowed_states_and_actions(n_node,m_star,index=True,terminal=terminal)
    print(allowed_actions,len(allowed_actions),len(allowed_states))
    

    def T_action(action):
        print(action)
        #print(get_action_from_index(n_node,action))

        T_a = np.zeros([len(allowed_states),len(allowed_states)])

        for s_old in allowed_states:
            t_a = get_transition_probs(n_node,m_star,get_state_from_index(n_node,m_star,s_old),get_action_from_index(n_node,action),p_l,p_bm,trials=trials,states=allowed_states)
            #print(get_state_from_index(s_old))
            #print(np.where(t_a!=0)[0])
            for s_new in np.where(t_a!=0)[0]:
                # When we care only about the transient block of the transition matrices, we can ignore all 
                # transitions that take us to a terminal state
                if not terminal and is_terminal_state(n_node,get_state_from_index(n_node,m_star,s_new)): 
                    continue
                else:
                    if is_allowed_state(n_node,get_state_from_index(n_node,m_star,s_new)):
                        T_a[np.where(allowed_states==s_new)[0][0]][np.where(allowed_states==s_old)[0][0]] = t_a[s_new]

        #print("T_a(s',s):")
        #print(np.round(T_a,2))
        return T_a
    
    T = Parallel(n_jobs=n_jobs)(delayed(T_action)(action) for action in allowed_actions)
#     T = []  
#     for action in allowed_actions:
#         print(action)
#         #print(get_action_from_index(n_node,action))

#         T_a = np.zeros([len(allowed_states),len(allowed_states)])

#         for s_old in allowed_states:
#             t_a = get_transition_probs(n_node,m_star,get_state_from_index(n_node,m_star,s_old),get_action_from_index(n_node,action),p_l,p_bm,trials=trials,states=allowed_states)
#             #print(get_state_from_index(s_old))
#             #print(np.where(t_a!=0)[0])
#             for s_new in np.where(t_a!=0)[0]:
#                 # When we care only about the transient block of the transition matrices, we can ignore all 
#                 # transitions that take us to a terminal state
#                 if not terminal and is_terminal_state(n_node,get_state_from_index(n_node,m_star,s_new)): 
#                     continue
#                 else:
#                     if is_allowed_state(n_node,get_state_from_index(n_node,m_star,s_new)):
#                         T_a[np.where(allowed_states==s_new)[0][0]][np.where(allowed_states==s_old)[0][0]] = t_a[s_new]

#         #print("T_a(s',s):")
#         #print(np.round(T_a,2))
#         T.append(np.round(T_a,2))
        #print("")
        
    return T
    

#T = []
#for action in allowed_actions:
#    print("action:")
#    print(get_action_from_index(action))
    
#    T_a = np.zeros([len(allowed_states),len(allowed_states)])
    
#    for s_old in allowed_states:
#        t_a = get_transition_probs(get_state_from_index(s_old),get_action_from_index(action),0.5,0.5)
#         print(get_state_from_index(s_old))
#         print(np.where(t_a!=0)[0])
#        for s_new in np.where(t_a!=0)[0]:
#            if is_allowed_state(get_state_from_index(s_new)):
#                T_a[np.where(allowed_states==s_new)[0][0]][np.where(allowed_states==s_old)[0][0]] = t_a[s_new]
    
#    print("T_a(s',s):")
#    print(np.round(T_a,2))
#    T.append(np.round(T_a,2))
#    print("")

#print("game over")

In [130]:
def the_meaning_of_it_all(action, T_a):
    print("action:")
    print("")
    print(str(action))
    print("\n \n")
    for i in range(len(allowed_states)):
        for j in range(len(allowed_states)):
            print("p = " + str(T_a[i][j])) 
            print("")
            print(str(get_state_from_index(allowed_states[j])) + "  --->  ")
            print("")
            print(str(get_state_from_index(allowed_states[i])))
            print("")
        print("\n")

In [135]:
#vary the index i from 0 to number of allowed actions to see all the transition probabilities 
i=10
the_meaning_of_it_all(get_action_from_index(allowed_actions[i]), T[i])

action:

[[0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 0.]]

 

p = 0.5

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]  --->  

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

p = 0.5

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]  --->  

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

p = 0.5

[[0. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 0.]
 [0. 1. 0. 0.]]  --->  

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

p = 0.5

[[0. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]]  --->  

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

p = 0.51

[[0. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]  --->  

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

p = 0.51

[[0. 0. 0. 1.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [1. 0. 0. 0.]]  --->  

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

p = 0.49

[[0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]  --->  

In [76]:
get_next_state([[0., 0., 1.],
 [0., 0., 0.],
 [1., 0., 0.]],[[0., 1., 0.],
 [1., 0., 0.],
 [0., 0., 0.]], 0.5,0.5)

array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 0.]])

In [14]:
[1,2,3]==0

False

In [29]:
T=get_transition_matrices(3,3,0.5,0.5,terminal=True)

[ 0  2  4  8 10] 5
0
2
4
8
10


In [82]:
T_transient=get_transition_matrices(4,2,0.5,0.5,terminal=False)

[ 0  2  4  8 10 16 18 20 32 34 36 40 42] 13
0
2
4
8
10
16
18
20
32
34
36
40
42


In [89]:
T_transient_43=get_transition_matrices(4,3,0.5,0.5,terminal=False)

[ 0  2  4  8 10 16 18 20 32 34 36 40 42] 13
0
2
4
8
10
16
18
20
32
34
36
40
42


In [92]:
T_transient_43[0].shape

(88, 88)

In [21]:
T_transient_44=get_transition_matrices(4,4,0.5,0.5,terminal=False)

[ 0  2  4  8 10 16 18 20 32 34 36 40 42] 13 165
0


KeyboardInterrupt: 

In [18]:
T_transient_52=get_transition_matrices(5,2,0.5,1,terminal=False)

[  0   2   4   8  10  16  18  20  32  34  36  40  42  64  66  68  72  74
  80  82  84 128 130 132 136 138 144 146 148 160 162 164 168 170] 34 175
0
2
4


KeyboardInterrupt: 

In [11]:
T_transient_53=get_transition_matrices(5,3,0.3,1,terminal=False)

[  0   2   4   8  10  16  18  20  32  34  36  40  42  64  66  68  72  74
  80  82  84 128 130 132 136 138 144 146 148 160 162 164 168 170] 34 505


In [8]:
def expected_waiting_time_LP(n_node,m_star,p,q,T=None,display=False):

    # Last modified: 8 January 2023
    
    '''
    Assuming fully homogeneous for now. The elementary link probability is p and the 
    swapping probability is q.
    '''

    states,actions=get_allowed_states_and_actions(n_node,m_star,index=True,terminal=False)
    
    D=len(states)
    A=len(actions)

    if T==None:
        print("Getting the transition matrices...\n")
        T=get_transition_matrices(n_node,m_star,p,q,terminal=False)

    v=get_initial_state_vector(n_node,m_star,p)
    
    W=cvx.Variable((D,A))
    
    x=np.sum([W[:,a] for a in range(A)],0)
    t=np.sum([T[a]@W[:,a] for a in range(A)],0)
    
    c=[W[:,a]>=0 for a in range(A)]
    c+=[x-t==v]

    obj=cvx.Minimize(cvx.sum(x))
    prob=cvx.Problem(obj,constraints=c)

    prob.solve(verbose=display,solver=cvx.ECOS)

    return prob.value, W.value, x.value, v



#########################################
#### Disregard the function below #######
#########################################
def expected_waiting_time_LP_alt(n_node,m_star,p,q,T=None,display=False):

    # Last modified: 8 January 2023
    
    '''
    Assuming fully homogeneous for now. The elementary link probability is p and the 
    swapping probability is q.
    '''

    states,actions=get_allowed_states_and_actions(n_node,m_star,index=True,terminal=False)
    
    D=len(states)
    A=len(actions)

    if T==None:
        T=get_transition_matrices(n_node,m_star,p,q,terminal=False)

    dim=T[0].shape[0]
        
    v=get_initial_state_vector(n_node,m_star,p)
    
    X=cvx.Variable((D,A))
    
    x=np.sum([X[:,a] for a in range(A)],0)
    
    Q=[inv(np.identity(dim)-T[a]) for a in range(A)]
    
    c=[X[:,a]>=0 for a in range(A)]
    c+=[x==v]

    obj=cvx.Minimize(cvx.sum([Q[a]@X[:,a] for a in range(A)]))
    prob=cvx.Problem(obj,constraints=c)

    prob.solve(verbose=display,solver=cvx.ECOS)

    return prob.value

In [9]:
wt=[]
for p_l in np.linspace(0.2,1.0,9):
    T_transient_53=get_transition_matrices(5,3,p_l,0.5,terminal=False)
    wt.append(expected_waiting_time_LP(5,3,p_l,0.5,T=T_transient_53,display=True)[0])

[  0   2   4   8  10  16  18  20  32  34  36  40  42  64  66  68  72  74
  80  82  84 128 130 132 136 138 144 146 148 160 162 164 168 170] 34 505
                                     CVXPY                                     
                                    v1.1.11                                    
(CVXPY) Feb 02 07:48:19 AM: Your problem has 17170 variables, 35 constraints, and 0 parameters.
(CVXPY) Feb 02 07:48:19 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 07:48:19 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 07:48:19 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Feb 02

11  +1.666e+01  +1.668e+01  +1e+01  2e-02  3e-04  2e-02  6e-04  0.5754  3e-01   1  1  1 |  0  0
12  +1.715e+01  +1.716e+01  +9e+00  2e-02  2e-04  1e-02  5e-04  0.3572  6e-01   1  1  1 |  0  0
13  +1.894e+01  +1.895e+01  +6e+00  1e-02  1e-04  5e-03  3e-04  0.5293  2e-01   2  1  1 |  0  0
14  +1.902e+01  +1.903e+01  +5e+00  9e-03  1e-04  4e-03  3e-04  0.2731  6e-01   2  1  1 |  0  0
15  +2.002e+01  +2.002e+01  +2e+00  4e-03  6e-05  1e-03  1e-04  0.6905  2e-01   2  1  1 |  0  0
16  +2.017e+01  +2.017e+01  +2e+00  4e-03  5e-05  1e-03  1e-04  0.3435  5e-01   1  1  1 |  0  0
17  +2.057e+01  +2.058e+01  +8e-01  1e-03  2e-05  4e-04  5e-05  0.7101  1e-01   1  1  1 |  0  0
18  +2.068e+01  +2.068e+01  +5e-01  9e-04  1e-05  2e-04  3e-05  0.6059  4e-01   2  1  1 |  0  0
19  +2.074e+01  +2.074e+01  +2e-01  4e-04  6e-06  9e-05  1e-05  0.6133  7e-02   2  1  1 |  0  0
20  +2.078e+01  +2.078e+01  +3e-02  6e-05  9e-07  1e-05  2e-06  0.9317  8e-02   1  1  1 |  0  0
21  +2.079e+01  +2.079e+01  +2e-03  4e-0

(CVXPY) Feb 02 08:00:49 AM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Feb 02 08:00:49 AM: Applying reduction Dcp2Cone
(CVXPY) Feb 02 08:00:49 AM: Applying reduction CvxAttr2Constr
(CVXPY) Feb 02 08:00:49 AM: Applying reduction ConeMatrixStuffing
(CVXPY) Feb 02 08:00:49 AM: Applying reduction ECOS
(CVXPY) Feb 02 08:00:49 AM: Finished problem compilation (took 2.358e-01 seconds).
(CVXPY) Feb 02 08:00:49 AM: (Subsequent compilations of this problem, using the same arguments, should take less time.)
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
(CVXPY) Feb 02 08:00:49 AM: Invoking solver ECOS to obtain a solution.

ECOS 2.0.7 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    

(CVXPY) Feb 02 08:04:58 AM: Optimal value: 6.787e+00
(CVXPY) Feb 02 08:04:58 AM: Compilation took 2.286e-01 seconds
(CVXPY) Feb 02 08:04:58 AM: Solver (including time spent in interface) took 3.268e-01 seconds
[  0   2   4   8  10  16  18  20  32  34  36  40  42  64  66  68  72  74
  80  82  84 128 130 132 136 138 144 146 148 160 162 164 168 170] 34 505
                                     CVXPY                                     
                                    v1.1.11                                    
(CVXPY) Feb 02 08:09:07 AM: Your problem has 17170 variables, 35 constraints, and 0 parameters.
(CVXPY) Feb 02 08:09:07 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 08:09:07 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 08:09:07 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
--------------------------------------------

11  +4.283e+00  +4.285e+00  +1e+01  1e-02  1e-04  2e-03  7e-04  0.2002  6e-01   1  1  1 |  0  0
12  +4.463e+00  +4.464e+00  +7e+00  6e-03  8e-05  1e-03  4e-04  0.5135  2e-01   1  1  1 |  0  0
13  +4.550e+00  +4.551e+00  +4e+00  3e-03  5e-05  4e-04  2e-04  0.6793  4e-01   1  1  1 |  0  0
14  +4.614e+00  +4.615e+00  +2e+00  2e-03  3e-05  2e-04  1e-04  0.5499  2e-01   1  1  1 |  0  0
15  +4.634e+00  +4.634e+00  +2e+00  1e-03  2e-05  1e-04  9e-05  0.5227  4e-01   1  1  1 |  0  0
16  +4.656e+00  +4.656e+00  +7e-01  5e-04  8e-06  4e-05  4e-05  0.7239  2e-01   1  1  1 |  0  0
17  +4.660e+00  +4.660e+00  +4e-01  3e-04  5e-06  3e-05  3e-05  0.5236  4e-01   1  1  1 |  0  0
18  +4.666e+00  +4.666e+00  +1e-01  1e-04  1e-06  7e-06  7e-06  0.8740  2e-01   1  1  1 |  0  0
19  +4.667e+00  +4.667e+00  +8e-02  6e-05  9e-07  4e-06  5e-06  0.5997  4e-01   1  1  1 |  0  0
20  +4.668e+00  +4.668e+00  +4e-02  3e-05  4e-07  2e-06  2e-06  0.6422  1e-01   1  1  1 |  0  0
21  +4.668e+00  +4.668e+00  +1e-02  9e-0

(CVXPY) Feb 02 08:21:36 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 08:21:36 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 08:21:36 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Feb 02 08:21:36 AM: Compiling problem (target solver=ECOS).
(CVXPY) Feb 02 08:21:36 AM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Feb 02 08:21:36 AM: Applying reduction Dcp2Cone
(CVXPY) Feb 02 08:21:36 AM: Applying reduction CvxAttr2Constr
(CVXPY) Feb 02 08:21:36 AM: Applying reduction ConeMatrixStuffing
(CVXPY) Feb 02 08:21:36 AM: Applying reduction ECOS
(CVXPY) Fe

In [13]:
wt=[]
for m_star in [1,2,3,4,5,6,7,8]:
    T_transient_53=get_transition_matrices(4,m_star,0.5,0.5,terminal=False)
    wt.append(expected_waiting_time_LP(4,m_star,0.5,0.5,T=T_transient_53,display=True)[0])

[ 0  2  4  8 10 16 18 20 32 34 36 40 42] 13 12
                                     CVXPY                                     
                                    v1.1.11                                    
(CVXPY) Feb 02 08:49:23 AM: Your problem has 156 variables, 14 constraints, and 0 parameters.
(CVXPY) Feb 02 08:49:23 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 08:49:23 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 08:49:23 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Feb 02 08:49:23 AM: Compiling problem (target solver=ECOS).
(CVXPY) Feb 02 08:49:23 AM: Reduction chain: Dc

(CVXPY) Feb 02 08:49:32 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 08:49:32 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 08:49:32 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Feb 02 08:49:32 AM: Compiling problem (target solver=ECOS).
(CVXPY) Feb 02 08:49:32 AM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> ECOS
(CVXPY) Feb 02 08:49:32 AM: Applying reduction Dcp2Cone
(CVXPY) Feb 02 08:49:32 AM: Applying reduction CvxAttr2Constr
(CVXPY) Feb 02 08:49:32 AM: Applying reduction ConeMatrixStuffing
(CVXPY) Feb 02 08:49:32 AM: Applying reduction ECOS
(CVXPY) Fe

[ 0  2  4  8 10 16 18 20 32 34 36 40 42] 13 276
                                     CVXPY                                     
                                    v1.1.11                                    
(CVXPY) Feb 02 08:50:18 AM: Your problem has 3588 variables, 14 constraints, and 0 parameters.
(CVXPY) Feb 02 08:50:18 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 08:50:18 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 08:50:18 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Feb 02 08:50:18 AM: Compiling problem (target solver=ECOS).
(CVXPY) Feb 02 08:50:18 AM: Reduction chain: 

(CVXPY) Feb 02 08:51:19 AM: Optimal value: 8.331e+00
(CVXPY) Feb 02 08:51:19 AM: Compilation took 6.107e-02 seconds
(CVXPY) Feb 02 08:51:19 AM: Solver (including time spent in interface) took 1.019e-01 seconds
[ 0  2  4  8 10 16 18 20 32 34 36 40 42] 13 624
                                     CVXPY                                     
                                    v1.1.11                                    
(CVXPY) Feb 02 08:53:17 AM: Your problem has 8112 variables, 14 constraints, and 0 parameters.
(CVXPY) Feb 02 08:53:17 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Feb 02 08:53:17 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Feb 02 08:53:17 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
-------------------------------------------------------------------------------
                                  Compilation                  

12  +7.897e+00  +7.898e+00  +4e+00  6e-03  7e-05  8e-04  3e-04  0.5417  2e-01   1  1  1 |  0  0
13  +8.047e+00  +8.048e+00  +2e+00  3e-03  3e-05  4e-04  2e-04  0.5828  1e-01   1  1  1 |  0  0
14  +8.062e+00  +8.062e+00  +2e+00  2e-03  3e-05  3e-04  1e-04  0.3596  5e-01   1  1  1 |  0  0
15  +8.107e+00  +8.107e+00  +8e-01  1e-03  2e-05  1e-04  7e-05  0.6513  3e-01   1  1  1 |  0  0
16  +8.132e+00  +8.132e+00  +4e-01  5e-04  7e-06  6e-05  3e-05  0.6851  2e-01   1  1  1 |  0  0
17  +8.139e+00  +8.139e+00  +2e-01  3e-04  4e-06  3e-05  2e-05  0.8900  5e-01   1  1  1 |  0  0
18  +8.144e+00  +8.144e+00  +7e-02  1e-04  1e-06  8e-06  6e-06  0.7104  4e-02   1  1  1 |  0  0
19  +8.146e+00  +8.146e+00  +2e-02  3e-05  4e-07  3e-06  2e-06  0.7118  5e-02   1  1  1 |  0  0
20  +8.146e+00  +8.146e+00  +2e-02  3e-05  4e-07  2e-06  2e-06  0.2432  7e-01   1  1  1 |  0  0
21  +8.147e+00  +8.147e+00  +1e-02  2e-05  2e-07  1e-06  1e-06  0.8059  4e-01   1  1  1 |  0  0
22  +8.147e+00  +8.147e+00  +3e-03  4e-0

In [11]:
#p_sw = 1.0 n=5 m^star = 3
[19.19440520839969,
 9.362726209742217,
 5.995133961861097,
 4.409456146623453,
 3.571494330397827,
 3.0385548805783547,
 2.6604806913740284,
 2.3298579258031626,
 1.9999999999976656]

[19.19440520839969,
 9.362726209742217,
 5.995133961861097,
 4.409456146623453,
 3.571494330397827,
 3.0385548805783547,
 2.6604806913740284,
 2.3298579258031626,
 1.9999999999976656]

In [12]:
#p_sw = 0.5 n=5 m^star = 3
[47.84232387231571,
 20.786290007888468,
 12.340365434020557,
 8.788711512412124,
 6.787087933871474,
 5.511120062532887,
 4.66799463586661,
 4.039913479278675,
 3.61780104638825]

[47.84232387231571,
 20.786290007888468,
 12.340365434020557,
 8.788711512412124,
 6.787087933871474,
 5.511120062532887,
 4.66799463586661,
 4.039913479278675,
 3.61780104638825]